### Setup (Colab)
Run the following cell to install the package and dependencies if running in Google Colab.
If running locally, ensure you have installed the package via `pip install -e .`

In [ ]:
!git clone https://github.com/zheliu17/nanoRecSys.git
%pip install -q -e ./nanoRecSys[llm]

import psutil  # noqa: F401

# In fact, we don't need psutil. force-reinstall to trigger colab restart
%pip install --force-reinstall psutil=={psutil.__version__}
print("Installation complete. Please restart runtime...")

### 1. Generate Embeddings, Negative Mining

Make sure `item_tower.pth` and `user_tower.pth` are in the `artifacts` directory (`nanoRecSys.config.settings.artifacts_dir`).

See [Sequential Transformer Training](./sequential_transformer.ipynb) or [Metaflow Pipeline](../pipeline.py) for details on how to train/download the checkpoints.

In [ ]:
# Unsloth should be import as the first package
import nanoRecSys.indexing.build_embeddings
from nanoRecSys.training.mine_negatives_sasrec import run_pipeline

nanoRecSys.indexing.build_embeddings.build_item_embeddings(batch_size=128)
nanoRecSys.indexing.build_embeddings.build_user_embeddings(batch_size=128)

# This will pair each interaction in the training set with one hard negative.
# We don't need 20M interactions for training the LLM ranker; 0.1 sampling_ratio is absolutely sufficient.
run_pipeline(
    batch_size=128,
    top_k=100,
    skip_top=0,
    sampling_ratio=0.1,
    suffix="llm_ranker",
    save_history=True,
    num_random_negatives=2,
    save_embeddings=False,  # LLM ranker donesn't use user embeddings
)

### 2. Two Stage LLM Ranker Training

**Stage 1**: Embedding alignment. Projection training, no LoRA

**Stage 2**: Co-adaptation. Train projection and LLM together with LoRA.

About 10000 steps in totol. Takes about 2-3 hours on one A100 GPU.



Two stage training is recommended when total training steps is small (e.g., 10000) to stabilize the training. If you have access to better compute, consider training for more steps (e.g., 800000), larger batch size (e.g., 64) and direct fine-tuning for better performance.

In [ ]:
from nanoRecSys.config import settings
from nanoRecSys.training.train_llm_ranker import LLMTrainingConfig, main as train_main

config = LLMTrainingConfig(
    n_samples=int(10000),
    batch_size=32,
    learning_rate=1e-4,
    resume_from_checkpoint=False,
    warmup_steps=100,
    use_lora=False,
    report_to="none",  # i.e. 'wandb'
)

# about 1200 steps; enough for loss stabilization
train_main(config)

In [ ]:
# Set your projection weights path (*.pth); default is settings.llm_output_dir / "projection.pth"
projection_path = str(settings.llm_output_dir / "projection.pth")

config = LLMTrainingConfig(
    n_samples=int(80000),
    batch_size=32,
    learning_rate=5e-5,
    projection_path=projection_path,
    warmup_steps=1000,
    use_lora=True,
    positive_sample_ratio=1,
    report_to="none",  # i.e. 'wandb'
)

train_main(config)

### 3. Evaluation

In [ ]:
from nanoRecSys.eval.eval_llm_rankers import evaluate
from nanoRecSys.utils.utils import format_results_to_dataframe

results = evaluate(
    method="local",  # Evaluate the LLM ranker
    num_users=100,
    local_batch_size=16,
    use_lora=True,
    adapter_path=str(settings.llm_output_dir),
)

format_results_to_dataframe(results)